# Gomoku AlphaZero: Parallel Training (Kaggle T4 x2)

Optimized for **Kaggle's Dual-T4 GPU + 4-Core CPU**.

### 🚀 Optimizations
- **Multi-threading self-play**: 3 background threads collect game data concurrently.
- **Dual-GPU training**: `nn.DataParallel` splits large batches (512+) across both T4s.
- **Zero-copy MCTS**: In-place `do_move`/`undo_move` — no `deepcopy` overhead.
- **Weight sync**: Main thread periodically pushes updated model weights to workers.
- **High-density logging**: Every game and every train batch printed instantly.

In [ ]:
import os, time, random, threading
import numpy as np
from collections import deque, defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from dataclasses import dataclass, field
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython import display as ipydisplay

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## 1. Config

In [ ]:
@dataclass
class Config:
    board_size:   int   = 15
    n_in_row:     int   = 5
    n_res_blocks: int   = 10
    n_filters:    int   = 256
    n_workers:    int   = 3
    c_puct:            float = 5.0
    n_playout:         int   = 400
    n_playout_eval:    int   = 800
    pure_mcts_playout: int   = 1000
    learn_rate:       float = 2e-3
    train_batch_size: int   = 512
    buffer_size:      int   = 20000
    update_epochs:    int   = 5
    kl_coeff:         float = 0.02
    weight_sync_freq: int   = 10
    total_iterations: int   = 5000
    check_freq:       int   = 50
    n_eval_games:     int   = 6
    checkpoint_dir:   str   = "./checkpoints"
    device: object = field(default=None, init=False)
    n_gpus: int    = field(default=0,    init=False)

    def __post_init__(self):
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
            self.n_gpus = torch.cuda.device_count()
        else:
            self.device = torch.device("cpu")
            self.n_gpus = 0
        os.makedirs(self.checkpoint_dir, exist_ok=True)
        print(f"Config OK — device={self.device}, n_gpus={self.n_gpus}")

cfg = Config()

## 2. Environment

In [ ]:
class GomokuEnv:
    def __init__(self, board_size=15, n_in_row=5):
        self.board_size = board_size
        self.n_in_row   = n_in_row
        self.reset()

    def reset(self):
        self.board      = np.zeros((self.board_size, self.board_size), dtype=np.int8)
        self.history    = []
        self.availables = set(range(self.board_size ** 2))
        self.current_player = 1  # 1=Black, 2=White

    def do_move(self, move):
        r, c = divmod(move, self.board_size)
        self.board[r, c] = self.current_player
        self.availables.remove(move)
        self.history.append((self.current_player, move))
        self.current_player = 3 - self.current_player

    def undo_move(self):
        prev_player, move = self.history.pop()
        r, c = divmod(move, self.board_size)
        self.board[r, c] = 0
        self.availables.add(move)
        self.current_player = prev_player

    def current_state(self):
        s = np.zeros((4, self.board_size, self.board_size), dtype=np.float32)
        s[0] = (self.board == self.current_player)
        s[1] = (self.board == (3 - self.current_player))
        if self.history:
            lm = self.history[-1][1]
            s[2, lm // self.board_size, lm % self.board_size] = 1.0
        if len(self.history) % 2 == 0:
            s[3] = 1.0
        return s

    def game_ended(self):
        if len(self.history) < self.n_in_row * 2 - 1:
            return False, -1
        last_p, last_m = self.history[-1]
        h, w = divmod(last_m, self.board_size)
        for dh, dw in [(1,0),(0,1),(1,1),(1,-1)]:
            cnt = 1
            for sign in (1, -1):
                for step in range(1, self.n_in_row):
                    r, c = h + sign*step*dh, w + sign*step*dw
                    if 0 <= r < self.board_size and 0 <= c < self.board_size and self.board[r,c] == last_p:
                        cnt += 1
                    else:
                        break
            if cnt >= self.n_in_row:
                return True, last_p
        if not self.availables:
            return True, -1
        return False, -1

## 3. ResNet Model

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.ReLU(),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch)
        )
    def forward(self, x):
        return F.relu(x + self.net(x))

class Net(nn.Module):
    def __init__(self, size=15, n_res=10, ch=256):
        super().__init__()
        S = size
        self.body = nn.Sequential(
            nn.Conv2d(4, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.ReLU(),
            *[ResBlock(ch) for _ in range(n_res)]
        )
        self.p_head = nn.Sequential(
            nn.Conv2d(ch, 2, 1, bias=False), nn.BatchNorm2d(2), nn.ReLU(),
            nn.Flatten(), nn.Linear(2*S*S, S*S), nn.LogSoftmax(dim=1)
        )
        self.v_head = nn.Sequential(
            nn.Conv2d(ch, 1, 1, bias=False), nn.BatchNorm2d(1), nn.ReLU(),
            nn.Flatten(), nn.Linear(S*S, 256), nn.ReLU(), nn.Linear(256, 1), nn.Tanh()
        )
    def forward(self, x):
        x = self.body(x)
        return self.p_head(x), self.v_head(x)

_n = Net(cfg.board_size, cfg.n_res_blocks, cfg.n_filters)
_p, _v = _n(torch.zeros(1, 4, cfg.board_size, cfg.board_size))
print(f"Net OK — policy: {_p.shape}, value: {_v.shape}")
del _n, _p, _v

## 4. MCTS

> **Bug fix**: Probabilities are computed in `float64` and explicitly renormalized after Dirichlet mixing to prevent `np.random.choice` from throwing `ValueError: probabilities do not sum to 1`.

In [ ]:
class TreeNode:
    __slots__ = ['parent','children','n','Q','P']
    def __init__(self, parent, p):
        self.parent, self.children, self.n, self.Q, self.P = parent, {}, 0, 0.0, p

    def select(self, c_puct):
        return max(self.children.items(),
                   key=lambda kv: kv[1].Q + c_puct * kv[1].P * (self.n**0.5) / (1 + kv[1].n))

    def backup(self, v):
        self.n += 1
        self.Q += (v - self.Q) / self.n
        if self.parent:
            self.parent.backup(-v)


class MCTSPlayer:
    def __init__(self, policy_fn, c_puct=5.0, n_playout=400, is_selfplay=False):
        self.policy      = policy_fn
        self.c_puct      = c_puct
        self.n_playout   = n_playout
        self.is_selfplay = is_selfplay
        self.root        = TreeNode(None, 1.0)

    def _playout(self, env):
        node, trail = self.root, []
        while node.children:
            a, node = node.select(self.c_puct)
            env.do_move(a)
            trail.append(a)
        end, winner = env.game_ended()
        if not end:
            probs, v = self.policy(env)
            for act, prob in probs:
                node.children[act] = TreeNode(node, prob)
        else:
            v = 0.0 if winner == -1 else (1.0 if winner == env.current_player else -1.0)
        node.backup(-v)
        for _ in trail:
            env.undo_move()

    def get_action(self, env, temp=1e-3):
        for _ in range(self.n_playout):
            self._playout(env)
        acts, visits = zip(*[(a, nd.n) for a, nd in self.root.children.items()])

        # --- FIX: use float64 and explicitly renormalize to avoid np.random.choice ValueError ---
        visits_arr = np.array(visits, dtype=np.float64)
        log_v = np.log(visits_arr + 1e-10)
        scaled = log_v / temp
        probs = np.exp(scaled - scaled.max())   # numerically stable softmax
        probs /= probs.sum()                    # explicit renormalization (float64)

        if self.is_selfplay:
            noise = np.random.dirichlet(0.3 * np.ones(len(probs)))
            probs = 0.75 * probs + 0.25 * noise
            probs /= probs.sum()                # renormalize again after Dirichlet mixing
        # --------------------------------------------------------------------------------------

        move = np.random.choice(acts, p=probs)
        self.root = self.root.children.get(move, TreeNode(None, 1.0))
        self.root.parent = None
        move_prob = np.zeros(env.board_size ** 2)
        move_prob[list(acts)] = probs
        return move, move_prob

## 5. Multi-threaded Self-Play Workers

> **Why `threading` not `multiprocessing`**: `spawn` mode (required on Kaggle) cannot pickle notebook-defined functions (`Can't get attribute 'self_play_task' on __main__`). `threading` avoids all pickling. numpy releases the GIL during array ops, enabling real parallel MCTS across threads.

In [ ]:
def get_equi_data(data, size):
    """8× data augmentation: 4 rotations × 2 flips."""
    ext = []
    for s, p, z in data:
        p2d = p.reshape(size, size)
        for k in [1, 2, 3, 4]:
            s_r = np.array([np.rot90(ch, k) for ch in s])
            p_r = np.flipud(np.rot90(np.flipud(p2d), k))
            ext.append((s_r, p_r.flatten(), z))
            ext.append((np.array([np.fliplr(ch) for ch in s_r]),
                        np.flipud(np.fliplr(p_r)).flatten(), z))
    return ext


class SelfPlayWorker(threading.Thread):
    """Daemon thread: continuously plays self-play games and writes to shared buffer."""

    def __init__(self, wid, cfg, data_buffer, buf_lock, log_fn, stop_event):
        super().__init__(daemon=True)
        self.wid         = wid
        self.cfg         = cfg
        self.data_buffer = data_buffer
        self.buf_lock    = buf_lock
        self.log         = log_fn
        self.stop_event  = stop_event
        self.game_count  = 0
        # Own CPU model — avoids GPU contention during training
        self._net    = Net(cfg.board_size, cfg.n_res_blocks, cfg.n_filters)
        self._net.eval()
        self._wlock  = threading.Lock()

    def update_weights(self, state_dict):
        with self._wlock:
            self._net.load_state_dict(state_dict)
            self._net.eval()

    def _policy_fn(self, board):
        t = torch.tensor(board.current_state(), dtype=torch.float32).unsqueeze(0)
        with self._wlock:
            with torch.no_grad():
                log_p, v = self._net(t)
        probs = np.exp(log_p.numpy().flatten())
        return [(m, float(probs[m])) for m in board.availables], float(v)

    def run(self):
        env    = GomokuEnv(self.cfg.board_size, self.cfg.n_in_row)
        player = MCTSPlayer(self._policy_fn, self.cfg.c_puct,
                            self.cfg.n_playout, is_selfplay=True)
        while not self.stop_event.is_set():
            env.reset()
            player.root = TreeNode(None, 1.0)
            states, mcts_probs, cur_players = [], [], []
            while not self.stop_event.is_set():
                temp = 1.0 if len(states) < 30 else 1e-3
                move, mp_ = player.get_action(env, temp)
                states.append(env.current_state())
                mcts_probs.append(mp_)
                cur_players.append(env.current_player)
                env.do_move(move)
                end, winner = env.game_ended()
                if end:
                    z = np.zeros(len(cur_players))
                    if winner != -1:
                        z[np.array(cur_players) == winner]  =  1.0
                        z[np.array(cur_players) != winner]  = -1.0
                    aug = get_equi_data(zip(states, mcts_probs, z), self.cfg.board_size)
                    with self.buf_lock:
                        self.data_buffer.extend(aug)
                        buf_len = len(self.data_buffer)
                    self.game_count += 1
                    res = 'Black' if winner==1 else ('White' if winner==2 else 'Tie')
                    self.log(f"[Worker {self.wid}] Game {self.game_count}: "
                             f"{len(states)} steps | Winner={res} | Buffer={buf_len}")
                    break

## 6. Training Pipeline

In [ ]:
class TrainPipeline:
    def __init__(self, cfg):
        self.cfg      = cfg
        self.buffer   = deque(maxlen=cfg.buffer_size)
        self.buf_lock = threading.Lock()
        self.stop     = threading.Event()
        self.net = Net(cfg.board_size, cfg.n_res_blocks, cfg.n_filters).to(cfg.device)
        if cfg.n_gpus > 1:
            self.net = nn.DataParallel(self.net)
            print(f"DataParallel across {cfg.n_gpus} GPUs")
        self.opt     = optim.Adam(self.net.parameters(), weight_decay=1e-4, lr=cfg.learn_rate)
        self.metrics = defaultdict(list)
        self.workers = []
        self.iter    = 0

    def _log(self, msg):
        print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

    def _cpu_state_dict(self):
        base = self.net.module if isinstance(self.net, nn.DataParallel) else self.net
        return {k: v.cpu().clone() for k, v in base.state_dict().items()}

    def _start_workers(self):
        sd = self._cpu_state_dict()
        for i in range(self.cfg.n_workers):
            w = SelfPlayWorker(i, self.cfg, self.buffer, self.buf_lock, self._log, self.stop)
            w.update_weights(sd)
            w.start()
            self.workers.append(w)
        self._log(f"Started {len(self.workers)} self-play worker threads.")

    def _sync_weights(self):
        sd = self._cpu_state_dict()
        for w in self.workers:
            w.update_weights(sd)
        self._log("[Sync] Worker weights updated.")

    def _train_step(self):
        with self.buf_lock:
            batch = random.sample(self.buffer, self.cfg.train_batch_size)
        s   = torch.tensor(np.array([d[0] for d in batch]), dtype=torch.float32, device=self.cfg.device)
        p_t = torch.tensor(np.array([d[1] for d in batch]), dtype=torch.float32, device=self.cfg.device)
        z_t = torch.tensor(np.array([d[2] for d in batch]), dtype=torch.float32, device=self.cfg.device).unsqueeze(1)
        self.net.train()
        old_log_p, _ = self.net(s)
        old_log_p = old_log_p.detach()
        kl = 0.0
        for epoch in range(self.cfg.update_epochs):
            self.opt.zero_grad()
            log_p, v = self.net(s)
            loss = F.mse_loss(v, z_t) - torch.mean(torch.sum(p_t * log_p, dim=1))
            loss.backward()
            self.opt.step()
            with torch.no_grad():
                new_log_p, _ = self.net(s)
                kl = torch.mean(torch.sum(torch.exp(old_log_p) * (old_log_p - new_log_p), dim=1)).item()
            if kl > self.cfg.kl_coeff * 4:
                break
        self.net.eval()
        return loss.item(), kl, epoch + 1

    def _evaluate(self):
        self._log("--- Starting Evaluation ---")

        def ai_policy(board):
            t = torch.tensor(board.current_state(), dtype=torch.float32, device=self.cfg.device).unsqueeze(0)
            with torch.no_grad():
                log_p, v = self.net(t)
            probs = np.exp(log_p.cpu().numpy().flatten())
            return [(m, float(probs[m])) for m in board.availables], float(v)

        def pure_policy(board):
            n = len(board.availables)
            return [(m, 1.0/n) for m in board.availables], 0.0

        ai   = MCTSPlayer(ai_policy,   c_puct=self.cfg.c_puct, n_playout=self.cfg.n_playout_eval)
        pure = MCTSPlayer(pure_policy, c_puct=5.0,             n_playout=self.cfg.pure_mcts_playout)
        env  = GomokuEnv(self.cfg.board_size, self.cfg.n_in_row)
        wins = 0.0
        for g in range(self.cfg.n_eval_games):
            env.reset()
            ai.root = pure.root = TreeNode(None, 1.0)
            players = {1: ai, 2: pure} if g % 2 == 0 else {1: pure, 2: ai}
            while True:
                move, _ = players[env.current_player].get_action(env)
                env.do_move(move)
                end, winner = env.game_ended()
                if end:
                    if winner == -1:              wins += 0.5; res = 'Tie'
                    elif players[winner] == ai:   wins += 1;   res = 'AI Win'
                    else:                                       res = 'AI Lose'
                    self._log(f"  Eval {g+1}/{self.cfg.n_eval_games}: {res}")
                    break
        wr = wins / self.cfg.n_eval_games
        self._log(f"--- Eval Done: Win Rate = {wr:.2%} ---")
        return wr

    def _plot(self):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(self.metrics['loss'])
        axes[0].set_title('Training Loss'); axes[0].set_xlabel('Iteration')
        axes[1].plot(self.metrics['win_rate'])
        axes[1].set_title('Win Rate vs Pure MCTS')
        axes[1].set_xlabel(f'× {self.cfg.check_freq} iterations')
        plt.tight_layout()
        path = os.path.join(self.cfg.checkpoint_dir, 'training_curve.png')
        plt.savefig(path, dpi=100); plt.close()
        ipydisplay.clear_output(wait=True)
        ipydisplay.display(ipydisplay.Image(filename=path))

    def run(self):
        self._start_workers()
        self._log("Waiting for buffer to fill...")
        try:
            while self.iter < self.cfg.total_iterations:
                with self.buf_lock:
                    buf_size = len(self.buffer)
                if buf_size < self.cfg.train_batch_size:
                    time.sleep(2)
                    continue
                loss, kl, n_ep = self._train_step()
                self.iter += 1
                self.metrics['loss'].append(loss)
                self._log(f"[Train] Iter {self.iter}: Loss={loss:.4f} | KL={kl:.4f} | "
                          f"ep={n_ep} | buf={buf_size}")
                if self.iter % self.cfg.weight_sync_freq == 0:
                    self._sync_weights()
                if self.iter % self.cfg.check_freq == 0:
                    wr = self._evaluate()
                    self.metrics['win_rate'].append(wr)
                    ckpt = os.path.join(self.cfg.checkpoint_dir, f"model_{self.iter}.pt")
                    torch.save(self.net.state_dict(), ckpt)
                    self._log(f"Checkpoint: {ckpt}")
                    self._plot()
        except KeyboardInterrupt:
            self._log("Interrupted — shutting down workers...")
        finally:
            self.stop.set()
            for w in self.workers:
                w.join(timeout=5)
            final = os.path.join(self.cfg.checkpoint_dir, "model_final.pt")
            torch.save(self.net.state_dict(), final)
            self._log(f"Final model saved: {final}")

## 7. Run Training

Press **■ Stop** or `Ctrl+C` to halt gracefully — model is auto-saved on exit.

In [ ]:
pipeline = TrainPipeline(cfg)
pipeline.run()